# Train mô hình phát hiện đánh nhau (Fight Clip Classifier)

Notebook này train mô hình phân loại clip 2 giây thành **đánh nhau / không đánh nhau** cho dự án giám sát hành vi bất thường.

- **Kiến trúc:** MobileNetV3-Small (đã train sẵn trên ImageNet) trích đặc trưng từng khung hình → hiệu đặc trưng giữa hai khung liên tiếp (tín hiệu chuyển động) → GRU hai chiều → 1 đầu ra.
- **Đầu vào:** 16 khung hình lấy mẫu 8 khung/giây, cả khung hình co về 224×224.
- **Kết quả:** file `fight_clip.pt` để chép vào thư mục `weights/` của dự án.

- **Dữ liệu:** 2300 clip, notebook tự tải, không cần tài khoản nào.

**Cách dùng:** chọn *Runtime → Change runtime type → T4 GPU*, rồi *Runtime → Run all*. Toàn bộ mất khoảng 45–60 phút (tải dữ liệu ~2 GB, trích khung hình, train 15 epoch). Cuối cùng trình duyệt tự tải về `fight_clip.pt` và hai biểu đồ.

In [ ]:
import torch, torchvision, cv2, sys
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__, "| opencv", cv2.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ - hãy bật GPU trong Runtime > Change runtime type")

## 1. Cấu hình

Notebook tự tải hai bộ dữ liệu:

| Bộ dữ liệu | Số clip | Nội dung | Nguồn tải |
| :--- | ---: | :--- | :--- |
| **Real Life Violence Situations (RLVS)** | 1000 bạo lực + 1000 bình thường | Clip đời thực lấy từ YouTube: đánh nhau ngoài đường, trong nhà; clip bình thường gồm thể thao, ăn uống, đi lại... | Bản sao trên Hugging Face (`34data/real-life-violence`), 4 file zip, tổng ~1,9 GB |
| **Surveillance Camera Fight Dataset** | 150 đánh nhau + 150 bình thường | Clip 2 giây từ camera giám sát | GitHub (`seymanurakti/fight-detection-surv-dataset`) |

Không cần sửa gì ở ô dưới để train. Muốn thêm dữ liệu của riêng bạn (ví dụ RWF-2000 đã xin được từ tác giả, chép lên Google Drive) thì thêm vào `EXTRA_SOURCES` một bộ ba `(tên, thư mục clip đánh nhau, thư mục clip bình thường)` và đặt `MOUNT_DRIVE = True`.

In [ ]:
SEED = 42
EPOCHS = 15
BATCH_SIZE = 8
LR_HEAD = 1e-3          # tốc độ học của GRU + lớp phân loại
LR_BACKBONE = 1e-4      # tốc độ học của MobileNetV3 (tinh chỉnh nhẹ)
WEIGHT_DECAY = 1e-4

USE_RLVS = True         # 2000 clip Real Life Violence Situations (tự tải ~1,9 GB)
USE_FIGHT_SURV = True   # 300 clip camera giám sát (tự tải ~90 MB)

MOUNT_DRIVE = False     # True nếu dữ liệu thêm nằm trên Google Drive
EXTRA_SOURCES = [
    # ("rwf2000_train", "/content/drive/MyDrive/RWF-2000/train/Fight", "/content/drive/MyDrive/RWF-2000/train/NonFight"),
    # ("rwf2000_val",   "/content/drive/MyDrive/RWF-2000/val/Fight",   "/content/drive/MyDrive/RWF-2000/val/NonFight"),
]
SAVE_TO_DRIVE_DIR = ""  # ví dụ "/content/drive/MyDrive/btl-httm" để lưu thêm một bản trọng số lên Drive

CACHE_DIR = "/content/frame_cache"
STORE_SIZE = 256        # kích thước khung hình lưu trong cache (cắt ngẫu nhiên rồi co về img_size khi train)
MAX_SECONDS = 5.0       # mỗi video chỉ lấy tối đa 5 giây đầu

In [ ]:
import os, subprocess
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if USE_FIGHT_SURV and not os.path.isdir("/content/fight-detection-surv-dataset"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/seymanurakti/fight-detection-surv-dataset.git",
                    "/content/fight-detection-surv-dataset"], check=True)
if USE_FIGHT_SURV:
    print("fight_surv | fight:", len(os.listdir("/content/fight-detection-surv-dataset/fight")),
          "| noFight:", len(os.listdir("/content/fight-detection-surv-dataset/noFight")))

In [ ]:
import zipfile, urllib.request

RLVS_DIR = "/content/rlvs"
RLVS_URL = "https://huggingface.co/datasets/34data/real-life-violence/resolve/main/real-life-violence_{:04d}.zip"

def download(url, dest, retries=3):
    for attempt in range(1, retries + 1):
        try:
            urllib.request.urlretrieve(url, dest + ".part")
            os.replace(dest + ".part", dest)
            return
        except Exception as exc:
            print(f"  lần {attempt} lỗi: {exc}")
    raise RuntimeError(f"Không tải được {url}")

if USE_RLVS:
    os.makedirs(RLVS_DIR, exist_ok=True)
    for part in range(1, 5):
        marker = os.path.join(RLVS_DIR, f".done_{part}")
        if os.path.exists(marker):
            continue
        zip_path = os.path.join(RLVS_DIR, f"part{part}.zip")
        print(f"Đang tải phần {part}/4 ...")
        download(RLVS_URL.format(part), zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(RLVS_DIR)
        os.remove(zip_path)
        open(marker, "w").close()
    n_v = len(os.listdir(os.path.join(RLVS_DIR, "Violence")))
    n_nv = len(os.listdir(os.path.join(RLVS_DIR, "NonViolence")))
    print("rlvs | Violence:", n_v, "| NonViolence:", n_nv)
    assert n_v >= 900 and n_nv >= 900, "Bộ RLVS tải về không đủ file, hãy chạy lại ô này"

## 2. Liệt kê video và chia tập

Chia 70% train / 15% validation / 15% test, riêng cho từng nguồn và từng lớp.

Trong các bộ dữ liệu này, nhiều clip đánh số liền nhau được cắt từ **cùng một video gốc**. Nếu chia ngẫu nhiên, các đoạn của cùng một video sẽ rơi vào cả train lẫn test và kết quả test sẽ cao giả tạo. Vì vậy notebook chia theo **khối số thứ tự liên tiếp** (70% đầu để train, 15% tiếp để validation, 15% cuối để test). Cách này giảm rò rỉ nhưng không loại bỏ hoàn toàn.

In [ ]:
import random, glob
import numpy as np

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv", ".mpg", ".mpeg")

import re

sources = []
if USE_RLVS:
    sources.append(("rlvs", os.path.join(RLVS_DIR, "Violence"), os.path.join(RLVS_DIR, "NonViolence")))
if USE_FIGHT_SURV:
    sources.append(("fight_surv", "/content/fight-detection-surv-dataset/fight", "/content/fight-detection-surv-dataset/noFight"))
sources += list(EXTRA_SOURCES)
assert sources, "Chưa bật nguồn dữ liệu nào"

def natural_key(path):
    # V_2.mp4 đứng trước V_10.mp4: giữ các clip cùng video gốc nằm cạnh nhau
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", os.path.basename(path))]

splits = {"train": [], "val": [], "test": []}
for name, fight_dir, normal_dir in sources:
    for label, folder in ((1, fight_dir), (0, normal_dir)):
        found = sorted((p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                        if p.lower().endswith(VIDEO_EXT)), key=natural_key)
        if not found:
            raise FileNotFoundError(f"Không thấy video nào trong {folder}")
        group = [(p, label, name) for p in found]
        n_train = int(0.70 * len(group)); n_val = max(1, int(0.15 * len(group)))
        splits["train"] += group[:n_train]
        splits["val"] += group[n_train:n_train + n_val]
        splits["test"] += group[n_train + n_val:]
        print(f"{name:14s} nhãn={label}  {len(found):5d} video")
for k, v in splits.items():
    print(f"{k:5s} {len(v):5d} video | đánh nhau: {sum(l for _, l, _ in v)}")

## 3. Định nghĩa mô hình

Ô này là **bản sao nguyên văn** của `src/models/fight_classifier.py` trong dự án. Hệ thống sẽ nạp trọng số bằng đúng lớp này, nên đừng sửa kiến trúc ở đây mà không sửa file trong dự án.

In [ ]:
"""Mô hình phân loại clip ngắn: đánh nhau / không đánh nhau.

Kiến trúc: MobileNetV3-Small trích đặc trưng từng khung hình -> ghép thêm hiệu đặc trưng
giữa hai khung liên tiếp (tín hiệu chuyển động) -> GRU hai chiều -> 1 logit.

File này được notebook `train_fight_colab.ipynb` nhúng nguyên văn, nên chỉ phụ thuộc
torch / torchvision / numpy / opencv. Sửa kiến trúc ở đây thì phải train lại.
"""
import math
from collections import deque
from typing import Optional

import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small

FEATURE_DIM = 576  # số kênh đầu ra của MobileNetV3-Small.features
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DEFAULT_CONFIG = {
    "num_frames": 16,     # số khung hình trong một clip
    "sample_fps": 8.0,    # lấy mẫu 8 khung/giây -> một clip dài 2 giây
    "img_size": 224,
    "hidden": 128,
    "dropout": 0.3,
}


class FightClipClassifier(nn.Module):
    def __init__(self, hidden: int = 128, dropout: float = 0.3, pretrained: bool = False):
        super().__init__()
        weights = "DEFAULT" if pretrained else None
        self.backbone = mobilenet_v3_small(weights=weights).features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gru = nn.GRU(FEATURE_DIM * 2, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden * 4, 1)

    def encode_frames(self, frames: torch.Tensor) -> torch.Tensor:
        """frames: (N, 3, H, W) đã chuẩn hoá ImageNet -> (N, 576)."""
        return self.pool(self.backbone(frames)).flatten(1)

    def classify_features(self, feats: torch.Tensor) -> torch.Tensor:
        """feats: (B, T, 576) -> logit (B,). Logit > 0 nghĩa là nghiêng về 'đánh nhau'."""
        motion = feats[:, 1:] - feats[:, :-1]
        motion = torch.cat([torch.zeros_like(feats[:, :1]), motion], dim=1)
        out, _ = self.gru(torch.cat([feats, motion], dim=2))
        pooled = torch.cat([out.mean(dim=1), out.max(dim=1).values], dim=1)
        return self.head(self.dropout(pooled)).squeeze(1)

    def forward(self, clips: torch.Tensor) -> torch.Tensor:
        """clips: (B, T, 3, H, W) -> logit (B,)."""
        b, t = clips.shape[:2]
        feats = self.encode_frames(clips.flatten(0, 1)).view(b, t, -1)
        return self.classify_features(feats)


def preprocess_frame(frame_bgr: np.ndarray, img_size: int) -> torch.Tensor:
    """Ảnh BGR của OpenCV -> tensor (3, S, S) đã chuẩn hoá. Co cả khung hình về hình vuông."""
    rgb = cv2.cvtColor(cv2.resize(frame_bgr, (img_size, img_size), interpolation=cv2.INTER_AREA),
                       cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(rgb).permute(2, 0, 1).float().div_(255.0)
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x - mean) / std


class FightClipInference:
    """Chạy mô hình theo thời gian thực: mỗi ~1/sample_fps giây trích đặc trưng một khung hình,
    giữ num_frames đặc trưng gần nhất và phân loại cửa sổ đó. Chi phí mỗi lần gọi là một lượt
    MobileNetV3-Small, không phụ thuộc số người trong khung hình."""

    def __init__(self, model_path: str, device: str = "cpu"):
        checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)
        self.config = {**DEFAULT_CONFIG, **checkpoint.get("config", {})}
        self.recommended_threshold = float(checkpoint.get("threshold", 0.5))
        self.device = torch.device(device if (device == "cpu" or torch.cuda.is_available()) else "cpu")
        self.model = FightClipClassifier(self.config["hidden"], self.config["dropout"])
        self.model.load_state_dict(checkpoint["state_dict"])
        self.model.to(self.device).eval()
        self.interval = 1.0 / float(self.config["sample_fps"])
        self.num_frames = int(self.config["num_frames"])
        self.features = deque(maxlen=self.num_frames)  # (timestamp, tensor 576)
        self.last_probability: Optional[float] = None

    def reset(self):
        self.features.clear()
        self.last_probability = None

    def update(self, frame_bgr: np.ndarray, timestamp: float) -> Optional[float]:
        """Đưa khung hình mới vào. Trả về xác suất đánh nhau của ~2 giây gần nhất,
        hoặc None khi chưa đủ dữ liệu liên tục."""
        if not math.isfinite(timestamp):
            return self.last_probability
        if self.features:
            gap = timestamp - self.features[-1][0]
            if gap < 0 or gap > 4 * self.interval:
                self.reset()  # thời gian lùi hoặc đứt quãng: bỏ cửa sổ cũ
            elif gap < 0.9 * self.interval:
                return self.last_probability  # chưa đến lượt lấy mẫu
        x = preprocess_frame(frame_bgr, int(self.config["img_size"])).unsqueeze(0).to(self.device)
        with torch.no_grad():
            self.features.append((timestamp, self.model.encode_frames(x)[0]))
            if len(self.features) < self.num_frames:
                self.last_probability = None
            else:
                feats = torch.stack([f for _, f in self.features]).unsqueeze(0)
                self.last_probability = float(torch.sigmoid(self.model.classify_features(feats))[0])
        return self.last_probability

## 4. Trích khung hình ra cache

Mỗi video được đọc một lần: lấy mẫu 8 khung/giây, co cả khung hình về 256×256 và lưu thành file `.npy`. Việc train sau đó đọc từ cache nên nhanh hơn nhiều so với giải mã video ở mỗi epoch.

In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

CFG = dict(DEFAULT_CONFIG)
T, SAMPLE_FPS, IMG = CFG["num_frames"], CFG["sample_fps"], CFG["img_size"]
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(video):
    return os.path.join(CACHE_DIR, hashlib.md5(video.encode()).hexdigest() + ".npy")

def extract(video):
    out = cache_path(video)
    if os.path.exists(out):
        return True
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps != fps or fps < 1: fps = 25.0
    frames, next_t, index = [], 0.0, 0
    while True:
        ok, frame = cap.read()
        if not ok or index / fps > MAX_SECONDS: break
        if index / fps >= next_t - 1e-6:
            frames.append(cv2.cvtColor(cv2.resize(frame, (STORE_SIZE, STORE_SIZE), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB))
            next_t += 1.0 / SAMPLE_FPS
        index += 1
    cap.release()
    if len(frames) < 4:
        return False
    while len(frames) < T:           # clip quá ngắn: lặp lại khung cuối
        frames.append(frames[-1])
    np.save(out, np.stack(frames).astype(np.uint8))
    return True

all_videos = [p for split in splits.values() for p, _, _ in split]
with ThreadPoolExecutor(4) as pool:
    ok = list(tqdm(pool.map(extract, all_videos), total=len(all_videos), desc="Trích khung hình"))
bad = {v for v, good in zip(all_videos, ok) if not good}
print("Video lỗi/bị bỏ:", len(bad))
for k in splits:
    splits[k] = [it for it in splits[k] if it[0] not in bad]

## 5. Dataset và tăng cường dữ liệu

Khi train: chọn ngẫu nhiên một cửa sổ 16 khung liên tiếp, cắt ngẫu nhiên một vùng (cùng một vùng cho cả 16 khung), lật ngang, đổi độ sáng/tương phản. Khi đánh giá: lấy cửa sổ giữa video, không tăng cường — giống cách hệ thống xử lý lúc chạy thật (co cả khung hình về 224×224).

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

MEAN = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
STD = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)

class ClipDataset(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label, _ = self.items[i]
        frames = np.load(cache_path(path), mmap_mode="r")
        n = len(frames)
        start = random.randint(0, n - T) if self.train else (n - T) // 2
        x = torch.from_numpy(np.ascontiguousarray(frames[start:start + T])).permute(0, 3, 1, 2).float() / 255.0
        if self.train:
            scale = random.uniform(0.5, 1.0); ratio = random.uniform(0.75, 1.33)
            h = min(STORE_SIZE, int(STORE_SIZE * (scale / ratio) ** 0.5)); w = min(STORE_SIZE, int(STORE_SIZE * (scale * ratio) ** 0.5))
            top = random.randint(0, STORE_SIZE - h); left = random.randint(0, STORE_SIZE - w)
            x = x[:, :, top:top + h, left:left + w]
            if random.random() < 0.5: x = x.flip(3)
            x = ((x - 0.5) * random.uniform(0.8, 1.2) + 0.5 + random.uniform(-0.1, 0.1)).clamp(0, 1)
        x = F.interpolate(x, size=(IMG, IMG), mode="bilinear", align_corners=False)
        return (x - MEAN) / STD, torch.tensor(float(label))

loaders = {
    "train": DataLoader(ClipDataset(splits["train"], True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True),
    "val": DataLoader(ClipDataset(splits["val"], False), batch_size=BATCH_SIZE, num_workers=2),
    "test": DataLoader(ClipDataset(splits["test"], False), batch_size=BATCH_SIZE, num_workers=2),
}
xb, yb = next(iter(loaders["train"]))
print("Một batch:", tuple(xb.shape), "| nhãn:", yb.tolist())

## 6. Huấn luyện

Giữ lại trọng số của epoch có **độ chính xác validation cao nhất** (hoà thì lấy epoch có loss thấp hơn).

In [ ]:
import copy, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FightClipClassifier(CFG["hidden"], CFG["dropout"], pretrained=True).to(device)
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
    {"params": list(model.gru.parameters()) + list(model.head.parameters()), "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(loaders["train"]))
criterion = torch.nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

@torch.no_grad()
def predict(loader):
    model.eval()
    probs, labels, loss_sum = [], [], 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            logits = model(x)
        loss_sum += criterion(logits.float(), y).item() * len(y)
        probs += torch.sigmoid(logits.float()).cpu().tolist(); labels += y.cpu().tolist()
    return np.array(probs), np.array(labels), loss_sum / max(1, len(labels))

history = {k: [] for k in ("train_loss", "val_loss", "train_acc", "val_acc")}
best = {"acc": -1.0, "loss": float("inf"), "state": None, "epoch": 0}
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, loss_sum, correct, seen = time.time(), 0.0, 0, 0
    for x, y in loaders["train"]:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            logits = model(x)
            loss = criterion(logits.float(), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        loss_sum += loss.item() * len(y); correct += ((logits > 0).float() == y).sum().item(); seen += len(y)
    probs, labels, val_loss = predict(loaders["val"])
    val_acc = float(((probs >= 0.5) == labels).mean())
    history["train_loss"].append(loss_sum / seen); history["train_acc"].append(correct / seen)
    history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
    improved = (val_acc, -val_loss) > (best["acc"], -best["loss"])
    if improved:
        best.update(acc=val_acc, loss=val_loss, epoch=epoch, state=copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()}))
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {loss_sum/seen:.3f} acc {correct/seen:.3f} | val loss {val_loss:.3f} acc {val_acc:.3f} | {time.time()-t0:.0f}s {'<- tốt nhất' if improved else ''}")

model.load_state_dict(best["state"])
print(f"\nDùng trọng số của epoch {best['epoch']} (val acc {best['acc']:.3f})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, title in zip(axes, ("loss", "acc"), ("Loss", "Accuracy")):
    ax.plot(range(1, EPOCHS + 1), history["train_" + key], label="Train")
    ax.plot(range(1, EPOCHS + 1), history["val_" + key], label="Validation")
    ax.axvline(best["epoch"], color="gray", linestyle=":", label="Epoch được chọn")
    ax.set(xlabel="Epoch", title=title); ax.legend()
fig.tight_layout(); fig.savefig("fight_training_curves.png", dpi=150); plt.show()

## 7. Chọn ngưỡng và đánh giá trên tập test

Ngưỡng quyết định được chọn trên tập **validation** (ngưỡng cho F1 cao nhất), sau đó áp dụng **một lần** lên tập test. Con số trên tập test mới là con số nên đưa vào báo cáo.

In [ ]:
def metrics_at(probs, labels, thr):
    pred = probs >= thr
    tp = int((pred & (labels == 1)).sum()); fp = int((pred & (labels == 0)).sum())
    fn = int((~pred & (labels == 1)).sum()); tn = int((~pred & (labels == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return dict(threshold=float(thr), accuracy=(tp + tn) / max(1, len(labels)), precision=precision, recall=recall, f1=f1, tp=tp, fp=fp, fn=fn, tn=tn)

val_probs, val_labels, _ = predict(loaders["val"])
candidates = [metrics_at(val_probs, val_labels, t) for t in np.arange(0.30, 0.91, 0.05)]
chosen = max(candidates, key=lambda m: (m["f1"], m["precision"]))
print("Ngưỡng trên validation:")
for m in candidates:
    print(f"  {m['threshold']:.2f}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}{'  <- chọn' if m is chosen else ''}")
THRESHOLD = chosen["threshold"]

test_probs, test_labels, _ = predict(loaders["test"])
test_metrics = metrics_at(test_probs, test_labels, THRESHOLD)
print(f"\nTẬP TEST ({len(test_labels)} video, ngưỡng {THRESHOLD:.2f}):")
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in test_metrics.items()})

cm = np.array([[test_metrics["tn"], test_metrics["fp"]], [test_metrics["fn"], test_metrics["tp"]]])
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center", fontsize=16, color="white" if v > cm.max() / 2 else "black")
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Bình thường", "Đánh nhau"], yticklabels=["Bình thường", "Đánh nhau"],
       xlabel="Dự đoán", ylabel="Thực tế", title="Ma trận nhầm lẫn - tập test")
fig.tight_layout(); fig.savefig("fight_confusion_matrix.png", dpi=150); plt.show()

by_source = {}
for (path, label, src), p in zip(splits["test"], test_probs):
    by_source.setdefault(src, []).append((p >= THRESHOLD) == bool(label))
print("Độ chính xác test theo nguồn:", {k: f"{np.mean(v):.3f} ({len(v)} video)" for k, v in by_source.items()})

## 8. Lưu trọng số

File `fight_clip.pt` chứa trọng số, cấu hình mô hình, ngưỡng đã chọn và các chỉ số đánh giá.

**Đưa vào dự án:**
1. Chép `fight_clip.pt` vào thư mục `weights/` của dự án.
2. Trong `configs/config.yaml`, mục `behaviors.fighting.clip_model`, đặt `enabled: true`.
3. Chạy lại `python main.py`.

Nên chép thêm `fight_training_curves.png` và `fight_confusion_matrix.png` vào `docs/` để dùng trong báo cáo.

In [ ]:
checkpoint = {
    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
    "config": {k: (float(v) if isinstance(v, float) else int(v)) for k, v in CFG.items()},
    "threshold": float(THRESHOLD),
    "metrics": {k: float(v) for k, v in test_metrics.items()},
    "train_videos": len(splits["train"]),
}
torch.save(checkpoint, "fight_clip.pt")

# Kiểm tra lại: nạp bằng đúng lớp mà hệ thống sẽ dùng
runner = FightClipInference("fight_clip.pt", device="cpu")
sample = np.load(cache_path(splits["test"][0][0]))
for i in range(T):
    p = runner.update(cv2.cvtColor(cv2.resize(sample[i], (640, 360)), cv2.COLOR_RGB2BGR), i / SAMPLE_FPS)
print(f"Nạp lại thành công. Video test đầu tiên (nhãn {splits['test'][0][1]}): xác suất đánh nhau = {p:.3f}")
print("Kích thước file:", round(os.path.getsize("fight_clip.pt") / 1e6, 1), "MB")

if SAVE_TO_DRIVE_DIR:
    import shutil
    os.makedirs(SAVE_TO_DRIVE_DIR, exist_ok=True)
    for f in ("fight_clip.pt", "fight_training_curves.png", "fight_confusion_matrix.png"):
        shutil.copy(f, SAVE_TO_DRIVE_DIR)
    print("Đã lưu vào", SAVE_TO_DRIVE_DIR)

try:
    from google.colab import files
    for f in ("fight_clip.pt", "fight_training_curves.png", "fight_confusion_matrix.png"):
        files.download(f)
except ImportError:
    pass